# Notebook 02 — Reflection: the model checks and improves its own work

**Time:** about 40 minutes

**What you will learn**
- The reflection pattern: **write → critique → improve**
- Why *external feedback* (real test results) is stronger than self-critique
- How to stop a loop: a goal and a maximum number of tries

### The big idea
People rarely get things perfect on the first try. We write, re-read, and fix.
A model can do the same: we ask it to **critique** an answer, then **improve** it.

### ▶️ Run the two setup cells below first
They check that Ollama is ready and create `ask()`, the one helper we use to talk to the model.

In [1]:
# STEP 0 — Is Ollama running? Is the model downloaded?
import ollama

MODEL = "llama3.2"     # local model (bigger one: "qwen2.5:7b")

try:
    installed = [m.model for m in ollama.list().models]
    print("Models on this computer:", installed)
    if MODEL in installed or MODEL + ":latest" in installed:
        print("✅ Ready to go with", MODEL)
    else:
        print(f"⚠️ {MODEL} is missing. Run: ollama pull {MODEL}")
except ConnectionError:
    print("⚠️ Ollama is not running. Start it, then run this again:")
    print("   Arch Linux: sudo systemctl start ollama")
    print("   Windows 11: open Ollama from the Start menu")

Models on this computer: ['llama3.2:latest']
✅ Ready to go with llama3.2


In [2]:
# STEP 1 — A helper: send a message, get the answer back as text
DEFAULT_ROLE = "You are a helpful assistant. Keep answers short."

def ask(prompt, system=DEFAULT_ROLE):
    reply = ollama.chat(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        options={"temperature": 0},    # 0 = more predictable
    )
    return reply.message.content

print("ask() is ready.")

ask() is ready.


## Part 1 — Reflection on an email

Here is a weak first draft (from the course example). Problems: a vague date, a typo, and no name at the end.

In [3]:
draft_email = '''Hey Tommy,
I'll be in New York next month, let me know if
you'll be free for dinnr one night.'''

critique = ask("Review this email. List its problems as short "
               "bullet points (clarity, dates, spelling, "
               "missing parts).\n\n" + draft_email)
print("CRITIQUE:\n", critique)

CRITIQUE:
 Here are the problems with the email:

• Clarity: The sentence "I'll be in New York next month, let me know if you'll be free for dinnr one night" is unclear and could be misinterpreted. It would be better to say "I'll be in New York next month, would you like to meet for dinner one night?"
• Dates: The email does not specify the exact dates of the trip.
• Spelling: "dinnr" should be spelled as "dinner".
• Missing parts: The email does not include the sender's name, location, or any other relevant details.


In [4]:
better_email = ask(
    "Rewrite the email to fix every problem in the critique. "
    "Use the dates 5–7 June and sign it 'Sara'.\n\n"
    f"EMAIL:\n{draft_email}\n\nCRITIQUE:\n{critique}")
print("IMPROVED EMAIL:\n", better_email)

IMPROVED EMAIL:
 Here's a rewritten email that addresses the problems mentioned in the critique:

Subject: Dinner Invitation in New York (5-7 June)

Dear Tommy,

I'll be in New York from 5-7 June and was wondering if you'd like to meet for dinner one night during my stay. If you're free, please let me know and we can discuss a suitable time.

Looking forward to hearing from you.

Best,
Sara


### Check the result with code (not with the model)
Simple Python checks are **objective**: they give the same answer every time.

In [5]:
GREETINGS = ("hi", "hello", "dear", "hey")

def check_email(email):
    text = email.strip().lower()
    return {
        "has a greeting":   text.startswith(GREETINGS),
        "mentions June":    "june" in text,
        "signed by Sara": "sara" in text,
        "typo fixed":       "dinnr" not in text,
    }

print("Before:", check_email(draft_email))
print("After :", check_email(better_email))

Before: {'has a greeting': True, 'mentions June': False, 'signed by Sara': False, 'typo fixed': False}
After : {'has a greeting': False, 'mentions June': True, 'signed by Sara': True, 'typo fixed': True}


## Part 2 — Reflection on code, with real test results

Self-critique can miss bugs. Running the code gives **external feedback**: real error messages.

The plan:
1. The model writes a function `average(numbers)`.
2. **Our code** runs the tests and collects the errors.
3. The errors go back to the model, which fixes the code.
4. Stop when all tests pass, or after 3 tries.

⚠️ **Safety:** never run model-written code directly in your notebook. We run it in a **separate** Python process,
with a 5-second time limit, and we refuse code that uses files or the operating system.
This is a classroom safeguard, not a full sandbox.

In [6]:
import re, subprocess, sys

# The tests the code must pass (written by us, not by the model)
TESTS = """
assert average([2, 4, 6]) == 4, "average([2, 4, 6]) should be 4"
assert average([5]) == 5, "average([5]) should be 5"
assert average([]) == 0, "average([]) should be 0"
print("All tests passed!")
"""

CODE_BLOCK = r"```(?:python)?\s*(.*?)```"

def get_code(text):
    """Take the code out of a ```python ... ``` block."""
    match = re.search(CODE_BLOCK, text, re.DOTALL)
    if match:
        return match.group(1)
    return text

In [7]:
BLOCKED = ["import os", "import sys", "subprocess", "open(",
           "eval(", "exec(", "__import__", "shutil"]

def run_tests(code):
    """Run code + tests in a SEPARATE Python process."""
    for word in BLOCKED:
        if word in code:
            return False, f"Refused: the code contains '{word}'."
    command = [sys.executable, "-I", "-X", "utf8",
               "-c", code + "\n" + TESTS]
    try:
        result = subprocess.run(command, capture_output=True,
                                encoding="utf-8", timeout=5)
    except subprocess.TimeoutExpired:
        return False, "Stopped: it ran longer than 5 seconds."
    output = result.stdout + result.stderr
    return result.returncode == 0, output[-600:]

In [8]:
buggy = ("def average(numbers):\n"
         "    return sum(numbers) / len(numbers)")

passed, message = run_tests(buggy)
print("Passed:", passed)
print(message)

Passed: False
Traceback (most recent call last):
  File "<string>", line 6, in <module>
    assert average([]) == 0, "average([]) should be 0"
           ~~~~~~~^^^^
  File "<string>", line 2, in average
    return sum(numbers) / len(numbers)
           ~~~~~~~~~~~~~^~~~~~~~~~~~~~
ZeroDivisionError: division by zero



The buggy example fails on the empty list (`ZeroDivisionError`). Now let the model write — and fix — its own code.

In [9]:
task = ("Write a Python function average(numbers) that returns "
        "the average of a list. Only the code, no explanation.")
code = get_code(ask(task))
MAX_TRIES = 3

for attempt in range(1, MAX_TRIES + 1):
    passed, message = run_tests(code)
    print(f"--- Attempt {attempt} ---\n{code}\nResult: {message}\n")
    if passed:
        print("✅ Done: all tests pass.")
        break
    if attempt < MAX_TRIES:           # send the real error back
        fix_request = (f"This code fails its tests.\n\nCODE:\n{code}"
                       f"\n\nERROR:\n{message}\n\n"
                       "Fix it. Reply with only the corrected code.")
        code = get_code(ask(fix_request))

if not passed:
    print(f"❌ Still failing after {MAX_TRIES} tries.")

--- Attempt 1 ---
def average(numbers):
    return sum(numbers) / len(numbers)

Result: Traceback (most recent call last):
  File "<string>", line 7, in <module>
    assert average([]) == 0, "average([]) should be 0"
           ~~~~~~~^^^^
  File "<string>", line 2, in average
    return sum(numbers) / len(numbers)
           ~~~~~~~~~~~~~^~~~~~~~~~~~~~
ZeroDivisionError: division by zero


--- Attempt 2 ---
def average(numbers):
    if len(numbers) == 0:
        return 0
    return sum(numbers) / len(numbers)

Result: All tests passed!


✅ Done: all tests pass.


## Part 3 — When should a reflection loop stop?

A loop needs clear exits, or it may run forever (and waste time):

| Exit | Example |
|---|---|
| **Goal reached** | All tests pass |
| **Budget used** | 3 tries are over |
| **No progress** | The new version is not better than the old one |

👉 **Discuss:** which of these exits does the loop in Part 2 use?

## ✍️ Try it yourself
1. Add a test to `TESTS`: `assert average([1, 2]) == 1.5` and run Part 2 again.
2. In Part 1, add one more check to `check_email`, e.g. "mentions dinner".

## ✅ Summary
- Reflection = write → critique → improve.
- Feedback from **real checks** (tests, rules) is more reliable than the model judging itself.
- Every loop needs a goal and a maximum number of tries.

**Next:** Notebook 03 — giving the model **tools**.